# Generative Models in Practice: AE → VAE → CVAE

This notebook is structured around **what you'd actually do on a real project** — not definitions.

Each section answers a practical question:

| Part | Question | Model |
|------|----------|-------|
| 1 | How do I compress images into a small vector and get them back? | **Autoencoder** |
| 2 | How do I generate new realistic images from random noise? | **VAE** |
| 3 | How do I generate images of a *specific class* on demand? | **CVAE** |

> **Dataset**: MNIST (Parts 1 & 3) and dog images from Google Drive (Part 2).  
> Everything runs on CPU — GPU makes Part 2 faster but is not required.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


---
# Part 1 — Autoencoder: Compress & Reconstruct

## The practical use-case
Autoencoders are used when you need a **compact representation** of high-dimensional data:  
image compression, anomaly detection, pre-training for downstream tasks.

The model learns a bottleneck: encode the input to a small vector `z`, then decode back.

```
Input (784) ──► Encoder ──► z (32) ──► Decoder ──► Reconstruction (784)
```

The only training signal is: *how similar is the output to the input?*  
No labels needed — this is **unsupervised**.

> ⚠️ **Key limitation you'll discover**: the latent space is unstructured.  
> Sampling a random `z` produces garbage. That's the motivation for VAE in Part 2.


## Load MNIST

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False)

test_imgs, test_labels = next(iter(test_loader))

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    axes[i].imshow(test_imgs[i].squeeze(), cmap='gray')
    axes[i].set_title(str(test_labels[i].item()))
    axes[i].axis('off')
plt.suptitle("Sample MNIST digits")
plt.tight_layout()
plt.show()


## Build the Autoencoder

**Architecture**:
- Encoder: `784 → 256 → 64 → latent_dim`  
- Decoder: `latent_dim → 64 → 256 → 784`  
- Output activation: **Sigmoid** (pixel values live in [0, 1])

Fill in the `TODO` blanks to complete the model.


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim

        # TODO 1: Define the encoder.
        # Layers: Linear(784, 256) → ReLU → Linear(256, 64) → ReLU → Linear(64, latent_dim)
        self.encoder = nn.Sequential(
            # ↓ fill in here
        )

        # TODO 2: Define the decoder (mirror of encoder).
        # Layers: Linear(latent_dim, 64) → ReLU → Linear(64, 256) → ReLU → Linear(256, 784) → Sigmoid
        self.decoder = nn.Sequential(
            # ↓ fill in here
        )

    def forward(self, x):
        x_flat = x.view(-1, 784)
        z      = self.encoder(x_flat)   # TODO 3: encode to latent z
        recon  = self.decoder(z)        # TODO 4: decode back
        return recon, z


## Train the Autoencoder

Loss: **Binary Cross-Entropy** (BCE) between the reconstructed pixels and the originals.  
BCE works well here because pixel values are in [0, 1] (Sigmoid output).


In [ ]:
ae = Autoencoder(latent_dim=32).to(device)
ae_optimizer = optim.Adam(ae.parameters(), lr=1e-3)
ae_epochs = 10

for epoch in range(1, ae_epochs + 1):
    ae.train()
    total_loss = 0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        recon, _ = ae(imgs)

        # TODO 5: Compute reconstruction loss (BCE).
        # Tip: imgs must be flattened to (-1, 784) to match recon's shape.
        loss = F.binary_cross_entropy(recon, _________)

        ae_optimizer.zero_grad()
        loss._______()        # TODO 6: backpropagate
        ae_optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}/{ae_epochs}  |  Loss: {total_loss/len(train_loader):.4f}")


## Evaluate: Reconstruction quality

A well-trained AE should produce reconstructions that look almost identical to the originals.


In [ ]:
ae.eval()
with torch.no_grad():
    recon_imgs, _ = ae(test_imgs.to(device))
recon_imgs = recon_imgs.view(-1, 28, 28).cpu()

n = 10
fig, axes = plt.subplots(2, n, figsize=(15, 3))
for i in range(n):
    axes[0, i].imshow(test_imgs[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_imgs[i], cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=12)
axes[1, 0].set_ylabel("AE Recon", fontsize=12)
plt.suptitle("AE Reconstruction")
plt.tight_layout()
plt.show()


## The latent space problem — why AE can't generate

Sample a random `z` vector and decode it. Notice the output is noise or barely recognisable.

**Why?** The AE was only trained to reconstruct — it never learned *where* valid images live in latent space.  
The VAE in Part 2 fixes this by regularising `z` to follow N(0, 1).


In [ ]:
ae.eval()
with torch.no_grad():
    random_z  = torch.randn(10, 32).to(device)
    fake_imgs = ae.decoder(random_z).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    axes[i].imshow(fake_imgs[i], cmap='gray')
    axes[i].axis('off')
plt.suptitle("AE 'generation' from random z — mostly noise")
plt.tight_layout()
plt.show()

print("⚠️  The AE's latent space is unstructured — random samples don't decode well.")
print("   VAE regularises the latent space so every point decodes to something meaningful.")


---
# Part 2 — Variational Autoencoder: Generate New Images

## The practical use-case
VAEs are generative models: after training, you **sample `z ~ N(0,I)` and decode** to get a new,  
realistic image — no input image required. They're used for:

- Image synthesis and data augmentation
- Anomaly detection (anomalies have high reconstruction error)
- Learning smooth, interpolatable representations

## What's different from a plain AE?

| | AE | VAE |
|--|--|--|
| Encoder output | single vector `z` | distribution `(μ, σ²)` |
| Sampling | none | `z = μ + σ·ε`, `ε~N(0,I)` |
| Loss | Recon only | Recon + **KL divergence** |
| Latent space | unstructured | regularised ≈ N(0,1) |

The **reparameterisation trick** (`z = μ + σ·ε`) makes sampling differentiable so gradients flow through.

## Loss function

```
Total Loss = BCE(reconstruction, input) + KL(q(z|x) || N(0,1))
```

The KL term penalises the encoder for deviating from a standard normal — this is what structures the latent space.


## Load dog images from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class DogDataset(Dataset):
    def __init__(self, img_dir, transform1=None, transform2=None):
        self.img_dir    = img_dir
        self.img_names  = os.listdir(img_dir)
        self.transform1 = transform1
        self.transform2 = transform2

    def __getitem__(self, index):
        img = Image.open(os.path.join(self.img_dir, self.img_names[index]))
        if self.transform1: img = self.transform1(img)
        if self.transform2: img = self.transform2(img)
        return img

    def __len__(self):
        return len(self.img_names)

# Resize + crop first (deterministic), then augment (random)
transform1 = transforms.Compose([transforms.Resize(64), transforms.CenterCrop(64)])
transform2 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=10)], p=0.3),
    transforms.ToTensor(),
])

batch_size = 32
dog_dataset = DogDataset(
    img_dir='/content/drive/MyDrive/VAE/dog-images/all-dogs/all-dogs',
    transform1=transform1, transform2=transform2,
)
dog_loader = DataLoader(dog_dataset, batch_size=batch_size, shuffle=True,
                        num_workers=4, drop_last=True)

x_val = next(iter(dog_loader))
fig = plt.figure(figsize=(20, 6))
for i, img in enumerate(x_val[:16]):
    ax = fig.add_subplot(2, 8, i + 1, xticks=[], yticks=[])
    plt.imshow(img.numpy().transpose(1, 2, 0))
plt.suptitle("Sample dog images (64×64)")
plt.tight_layout()
plt.show()


## Build the Convolutional VAE

We use convolutional layers because images have spatial structure that fully-connected layers would miss.

**Encoder** downsamples 64×64 → 1×1, outputting `latent_dim*2` channels (μ and log σ² concatenated).  
**Decoder** upsamples 1×1 → 64×64 using transposed convolutions.

Fill in the `TODO` blanks. The shape comments tell you what each layer produces.


In [ ]:
from torch.autograd import Variable

class VAE(nn.Module):
    def __init__(self, latent_dim=32, no_of_sample=10, batch_size=32, channels=3):
        super().__init__()
        self.no_of_sample = no_of_sample
        self.batch_size   = batch_size
        self.channels     = channels
        self.latent_dim   = latent_dim

        # ── Encoder ──────────────────────────────────────────────────────
        def enc_block(n_in, n_out, k=4, s=2, p=1, bn=True):
            layers = [nn.Conv2d(n_in, n_out, k, s, p, bias=False)]
            if bn: layers.append(nn.BatchNorm2d(n_out))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        # TODO 7: Complete the encoder Sequential.
        # Shape: (channels,64,64)→(64,32,32)→(128,16,16)→(256,8,8)→(512,4,4)→(latent_dim*2,1,1)
        # Use enc_block for the first 4 stages. Apply BN on stages 3 & 4 only.
        # Final Conv2d: in=512, out=latent_dim*2, kernel=4, stride=1, pad=0  (collapses 4×4 → 1×1)
        self.encoder = nn.Sequential(
            *enc_block(channels,  64,  bn=_______),    # → (64, 32, 32)
            *enc_block(64,  128,  bn=_______),          # → (128, 16, 16)
            *enc_block(128, 256,  bn=_______),          # → (256,  8,  8)
            *enc_block(256, 512,  bn=_______),          # → (512,  4,  4)
            nn.Conv2d(_______, latent_dim * 2, _______, _______, _______, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # ── Decoder ──────────────────────────────────────────────────────
        def dec_block(n_in, n_out, k=4, s=2, p=1):
            return [
                nn.ConvTranspose2d(n_in, n_out, k, s, p, bias=False),
                nn.BatchNorm2d(n_out),
                nn.ReLU(inplace=True),
            ]

        # TODO 8: Complete the decoder Sequential (mirrors encoder in reverse).
        # Shape: (latent_dim,1,1)→(512,4,4)→(256,8,8)→(128,16,16)→(64,32,32)→(channels,64,64)
        # First dec_block uses k=4, s=1, p=0 to expand 1×1 → 4×4.
        # Final: ConvTranspose2d(64, channels, k=4, s=2, p=1) + Sigmoid
        self.decoder = nn.Sequential(
            *dec_block(latent_dim, _______, k=_______, s=_______, p=_______),  # → (512, 4, 4)
            *dec_block(_______, 256),           # → (256,  8,  8)
            *dec_block(256, _______),           # → (128, 16, 16)
            *dec_block(_______, 64),            # → (64,  32, 32)
            nn.ConvTranspose2d(64, _______, _______, _______, _______),          # → (channels, 64, 64)
            nn.________(),                      # constrain output to [0, 1]
        )

    def encode(self, x):
        h = self.encoder(x)
        # TODO 9: Split h along the channel axis — first half → mu, second half → logvar
        mu     = h[:, :_______, :, :]
        logvar = h[:, _______:, :, :]
        return mu, logvar

    def reparameterise(self, mu, logvar):
        if self.training:
            samples = []
            for _ in range(self.no_of_sample):
                # TODO 10: std = exp(0.5 * logvar)
                std = logvar._______(0.5).exp_()
                eps = Variable(std.data.new(std.size()).normal_())
                # TODO 11: z = mu + eps * std
                samples.append(eps._______(std).add_(mu))
            return samples
        else:
            return mu   # at inference, just use the mean

    def decode(self, z):
        return self.decoder(z).view(-1, 3 * 64 * 64)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        if self.training:
            return [self.decode(z_i) for z_i in z], mu, logvar
        else:
            return self.decode(z), mu, logvar

    def loss_function(self, recon_x, x, mu, logvar):
        x_flat = x.view(-1, 3 * 64 * 64)

        # TODO 12: BCE reconstruction loss.
        # At training time recon_x is a list — average BCE across all samples.
        if self.training:
            BCE = sum(F.binary_cross_entropy(r, x_flat) for r in recon_x) / _______
        else:
            BCE = F.binary_cross_entropy(recon_x, x_flat)

        # TODO 13: KL divergence.
        # Formula: -0.5 * sum(1 + logvar - mu² - exp(logvar))
        # Normalise by (batch_size × 3 × 64 × 64)
        KLD = -0.5 * torch.sum(1 + _______ - _______.pow(2) - _______.exp())
        KLD /= x.size(0) * _______

        return BCE + KLD


## Train the VAE

In [ ]:
lr         = 1e-3
epochs     = 50
latent_dim = 32

vae = VAE(latent_dim, batch_size=batch_size).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    vae.train()
    total_loss = 0
    for data in dog_loader:
        data = data.to(device)
        vae_optimizer.zero_grad()

        # TODO 14: Forward pass (returns list of recons, mu, logvar during training)
        recon, mu, logvar = vae(_________)

        # TODO 15: Compute VAE loss
        loss = vae.loss_function(_______, _______, _______, _______)

        loss.backward()
        vae_optimizer.step()
        total_loss += loss.item()

    # Show one reconstruction per epoch so you can watch quality improve
    vae.eval()
    with torch.no_grad():
        recon_img, _, _ = vae(x_val[:1].to(device))
        img = recon_img.view(3, 64, 64).cpu().numpy().transpose(1, 2, 0)
    print(f"Epoch {epoch}/{epochs}  |  Loss: {total_loss/len(dog_loader):.4f}")
    plt.imshow(img.clip(0, 1)); plt.axis('off')
    plt.title(f"Epoch {epoch} reconstruction"); plt.show()


## Evaluate: Reconstruction

After training, the VAE should reproduce the main features of each dog (colour, rough shape).  
It won't be pixel-perfect — that's expected. The VAE trades off sharpness for a structured latent space.


In [ ]:
vae.eval()
with torch.no_grad():
    recon_batch, mu, _ = vae(x_val.to(device))
recon_batch = recon_batch.view(-1, 3, 64, 64).cpu().numpy().transpose(0, 2, 3, 1)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0, i].imshow(x_val[i].numpy().transpose(1, 2, 0))
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_batch[i].clip(0, 1))
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("VAE Recon", fontsize=11)
plt.suptitle("VAE Reconstruction")
plt.tight_layout()
plt.show()


## Latent space interpolation

Because the VAE regularises its latent space, you can **linearly interpolate** between two image encodings  
and every intermediate point decodes to a realistic image.

This is one of the most practically useful properties: it lets you do smooth morphing and explore the data manifold.


In [ ]:
vae.eval()
with torch.no_grad():
    mu_batch, _ = vae.encode(x_val[:2].to(device))

mu1, mu2 = mu_batch[0], mu_batch[1]
steps = 10
walk  = torch.stack([mu1 + (mu2 - mu1) * (i / (steps - 1)) for i in range(steps)])

with torch.no_grad():
    walk_imgs = vae.decoder(walk).cpu().numpy().transpose(0, 2, 3, 1)

fig, axes = plt.subplots(1, steps, figsize=(20, 3))
for i, img in enumerate(walk_imgs):
    axes[i].imshow(img.clip(0, 1))
    axes[i].axis('off')
plt.suptitle("Latent space walk: dog A → dog B (each frame is a valid decode)")
plt.tight_layout()
plt.show()


## Generate new dogs from random noise

Sample `z ~ N(0, 1)` — the VAE's regularised latent space means these random points  
decode to plausible dog images, unlike the plain AE.


In [ ]:
vae.eval()
with torch.no_grad():
    random_z  = torch.randn(16, latent_dim, 4, 4).to(device)
    generated = vae.decoder(random_z).cpu().numpy().transpose(0, 2, 3, 1)

fig = plt.figure(figsize=(20, 6))
for i, img in enumerate(generated):
    ax = fig.add_subplot(2, 8, i + 1, xticks=[], yticks=[])
    plt.imshow(img.clip(0, 1))
plt.suptitle("VAE generated dogs — z ~ N(0,1), no input image used")
plt.tight_layout()
plt.show()


---
# Part 3 — Conditional VAE: Generate Images on Demand

## The practical use-case
A plain VAE generates data but you can't control **what** it generates.  
A CVAE lets you say *"give me a 7"* and it will produce one.

This is useful for:
- **Data augmentation**: generate more samples of an underrepresented class
- **Controlled synthesis**: produce specific outputs for testing or labelling pipelines

## How conditioning works

You give the model a **class label `c`** (as a one-hot vector) alongside the image.  
It's simply concatenated to the input at both encoder and decoder:

```
Encoder input:  [x_flattened | one_hot(c)]
Decoder input:  [z           | one_hot(c)]
```

At generation time, you skip the encoder entirely — just pick a label, sample `z ~ N(0,1)`, and decode.


## Build the CVAE

In [ ]:
NUM_CLASSES = 10
LATENT_DIM  = 16
IMG_DIM     = 784    # 28×28

class CVAE(nn.Module):
    def __init__(self, img_dim=IMG_DIM, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        self.latent_dim  = latent_dim
        self.num_classes = num_classes

        # TODO 16: Encoder body.
        # Input size = img_dim + num_classes  (image pixels concatenated with one-hot label)
        # Layers: Linear(img_dim+num_classes, 512) → ReLU → Linear(512, 256) → ReLU
        self.encoder = nn.Sequential(
            nn.Linear(_______ + _______, 512),
            nn.ReLU(),
            nn.Linear(_______, _______),
            nn.ReLU(),
        )

        # TODO 17: Two separate heads — one for μ, one for log σ² (both 256 → latent_dim)
        self.fc_mu     = nn.Linear(_______, _______)
        self.fc_logvar = nn.Linear(_______, _______)

        # TODO 18: Decoder.
        # Input size = latent_dim + num_classes  (z concatenated with one-hot label)
        # Layers: Linear(latent_dim+num_classes, 256) → ReLU → Linear(256, 512) → ReLU → Linear(512, img_dim) → Sigmoid
        self.decoder = nn.Sequential(
            nn.Linear(_______ + _______, 256),
            nn.ReLU(),
            nn.Linear(_______, _______),
            nn.ReLU(),
            nn.Linear(_______, img_dim),
            nn.Sigmoid(),
        )

    def encode(self, x, c):
        x_flat = x.view(-1, IMG_DIM)
        # TODO 19: Concatenate x_flat and c along dim=1, pass through encoder, return (mu, logvar)
        h = self.encoder(torch.cat([_______, _______], dim=_______))
        return self.fc_mu(_______), self.fc_logvar(_______)

    def reparameterise(self, mu, logvar):
        # TODO 20: std = exp(0.5 * logvar),  eps ~ N(0,I),  return mu + eps * std
        std = (_______ * logvar).exp()
        eps = torch.randn_like(_______)
        return _______ + eps * _______

    def decode(self, z, c):
        # TODO 21: Concatenate z and c along dim=1, pass through decoder
        return self.decoder(torch.cat([_______, _______], dim=1))

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z     = self.reparameterise(mu, logvar)
        recon = self.decode(z, _______)    # TODO 22: don't forget to pass c
        return recon, mu, logvar

    @staticmethod
    def loss_function(recon, x, mu, logvar):
        # TODO 23: BCE with reduction='sum'
        BCE = F.binary_cross_entropy(recon, x.view(-1, IMG_DIM), reduction=_______)

        # TODO 24: KL divergence with reduction='sum', normalised by batch size
        KLD = -0.5 * torch.sum(_______ + logvar - _______ - _______)

        return (BCE + KLD) / x.size(0)

def to_onehot(labels, num_classes=NUM_CLASSES):
    return F.one_hot(labels, num_classes).float()


## Train the CVAE

In [ ]:
cvae = CVAE().to(device)
cvae_optimizer = optim.Adam(cvae.parameters(), lr=1e-3)
cvae_epochs = 20

for epoch in range(1, cvae_epochs + 1):
    cvae.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)

        # TODO 25: Convert integer labels to one-hot and move to device
        c = to_onehot(_______).to(device)

        # TODO 26: Forward pass — CVAE needs both image and condition c
        recon, mu, logvar = cvae(_______, _______)

        # TODO 27: Compute loss
        loss = CVAE.loss_function(_______, _______, _______, _______)

        cvae_optimizer.zero_grad()
        loss.backward()
        cvae_optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}/{cvae_epochs}  |  Loss: {total_loss/len(train_loader):.2f}")


## Evaluate: Reconstruction

The CVAE receives the image AND its label — reconstruction should be sharp.


In [ ]:
cvae.eval()
with torch.no_grad():
    c_test = to_onehot(test_labels[:10]).to(device)
    recon, _, _ = cvae(test_imgs[:10].to(device), c_test)
recon = recon.view(-1, 28, 28).cpu()

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(test_imgs[i].squeeze(), cmap='gray')
    axes[0, i].set_title(str(test_labels[i].item()))
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i], cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("CVAE Recon", fontsize=10)
plt.suptitle("CVAE Reconstruction")
plt.tight_layout()
plt.show()


## Generate one of each digit on demand

This is the payoff of conditioning: **sample random `z`, specify the label, get that digit.**  
No input image needed.


In [ ]:
cvae.eval()
with torch.no_grad():
    z = torch.randn(10, LATENT_DIM).to(device)
    labels_to_gen = torch.arange(10)
    c = to_onehot(labels_to_gen).to(device)
    generated = cvae.decode(z, c).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    axes[i].imshow(generated[i], cmap='gray')
    axes[i].set_title(f"'{i}'")
    axes[i].axis('off')
plt.suptitle("CVAE controlled generation — one sample per digit class")
plt.tight_layout()
plt.show()


## Generate multiple samples of the same digit

Because `z` is sampled randomly and `c` is fixed, each run produces a **different style** of the same digit.  
This is directly useful for **data augmentation** — you control class, not exact appearance.


In [ ]:
target_digit = 3   # ← change this to generate any digit 0–9

cvae.eval()
with torch.no_grad():
    z = torch.randn(10, LATENT_DIM).to(device)
    c = to_onehot(torch.full((10,), target_digit, dtype=torch.long)).to(device)
    imgs = cvae.decode(z, c).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i in range(10):
    axes[i].imshow(imgs[i], cmap='gray')
    axes[i].axis('off')
plt.suptitle(f"CVAE: 10 generated samples of digit '{target_digit}' (change target_digit above)")
plt.tight_layout()
plt.show()


---
# Part 4 — GAN: Adversarial Image Generation

## The practical use-case
GANs (Generative Adversarial Networks) generate **sharper, more realistic images** than VAEs.  
They're used for:

- **Image synthesis**: generate photorealistic faces, scenes, objects
- **Image-to-image translation**: horse → zebra, sketch → photo (CycleGAN, pix2pix)
- **Data augmentation**: create training images for rare classes
- **Super-resolution**: upscale low-res images

## How it works — two networks in competition

| Network | Job | Wins when... |
|---------|-----|--------------|
| **Generator G** | Takes random noise `z`, outputs a fake image | Discriminator can't tell it's fake |
| **Discriminator D** | Looks at an image, outputs P(real) | It correctly spots fakes |

They train against each other in a **minimax game**:

```
D wants:  D(real image) → 1   and   D(G(z)) → 0
G wants:  D(G(z)) → 1   (fool the discriminator)
```

The generator never sees real images directly — it only learns from the discriminator's feedback.

## VAE vs GAN — when to use which

| | VAE | GAN |
|--|--|--|
| Output quality | Blurry but stable | Sharp but can be unstable |
| Latent space | Structured, smooth | Less interpretable |
| Training | Stable | Can fail (mode collapse, vanishing gradients) |
| Generation control | Via CVAE conditioning | Via conditional GAN |
| Best for | Reconstruction, anomaly detection, interpolation | High-quality image synthesis |


## The Loss Functions

**Discriminator loss** — standard binary cross-entropy:
```
L_D = -[log D(x_real) + log(1 - D(G(z)))]
```
Maximise: correctly label real as 1, fake as 0.

**Generator loss (non-saturating version)** — this is the practical trick from the original paper:
```
L_G = -log D(G(z))        ← used in practice
```
NOT the naive `log(1 - D(G(z)))` which **saturates early** (gradient → 0 when D is winning).  
The non-saturating form gives strong gradients even when the generator is weak.

> **Why does the naive loss fail?**  
> Early in training D easily classifies fakes → `D(G(z)) ≈ 0` → `log(1 - 0) ≈ 0` → no gradient for G.  
> The non-saturating form `−log D(G(z))` avoids this: when `D(G(z)) ≈ 0`, the gradient is large.


## DCGAN Architecture

We implement **DCGAN** (Deep Convolutional GAN) — the standard baseline for image GANs.

**Generator**: `z (noise) → ConvTranspose2d layers → image`  
Upsamples from a latent vector to a full image using transposed convolutions.

**Discriminator**: `image → Conv2d layers → scalar`  
Downsamples the image and outputs a single probability (real vs fake).

Key DCGAN design rules:
- No pooling layers — use strided convolutions (D) and transposed convolutions (G)
- BatchNorm in both G and D (except G output and D input layers)
- LeakyReLU in D, ReLU in G, Tanh on G output
- Output images normalised to `[-1, 1]` (not `[0, 1]` like VAE)


## Setup — reuse the dog dataset from Part 2

In [ ]:
# Images for GAN training should be normalised to [-1, 1]  (Tanh output range)
# We rebuild the dog loader with the correct normalisation.

transform_gan = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # [0,1] → [-1,1]
])

# Reuse the DogDataset class defined in Part 2
dog_dataset_gan = DogDataset(
    img_dir='/content/drive/MyDrive/VAE/dog-images/all-dogs/all-dogs',
    transform1=transforms.Compose([transforms.Resize(64), transforms.CenterCrop(64)]),
    transform2=transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]),
)
gan_loader = DataLoader(dog_dataset_gan, batch_size=32, shuffle=True,
                        num_workers=4, drop_last=True)

# Fixed noise for monitoring generation quality over training
LATENT_DIM_GAN = 100
fixed_noise = torch.randn(16, LATENT_DIM_GAN, 1, 1).to(device)

print("GAN dataset ready:", len(dog_dataset_gan), "images")


## Build the Generator

The generator takes a noise vector `z` of shape `(latent_dim, 1, 1)` and upsamples  
through transposed convolutions to a `(3, 64, 64)` image.

Shape flow: `(100,1,1) → (512,4,4) → (256,8,8) → (128,16,16) → (64,32,32) → (3,64,64)`


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, channels=3, feature_maps=64):
        super().__init__()
        # feature_maps controls the base width — each stage doubles/halves it

        # TODO 28: Complete the generator.
        # Use nn.ConvTranspose2d + nn.BatchNorm2d + nn.ReLU for all hidden layers.
        # Final layer: ConvTranspose2d → Tanh (output in [-1, 1], no BatchNorm)
        #
        # Shape flow (k=kernel, s=stride, p=padding):
        #   (latent_dim, 1, 1)  → ConvTranspose2d(latent_dim, fm*8, k=4, s=1, p=0) → (fm*8, 4, 4)
        #   (fm*8, 4, 4)        → ConvTranspose2d(fm*8,  fm*4,  k=4, s=2, p=1)      → (fm*4, 8, 8)
        #   (fm*4, 8, 8)        → ConvTranspose2d(fm*4,  fm*2,  k=4, s=2, p=1)      → (fm*2, 16,16)
        #   (fm*2, 16,16)       → ConvTranspose2d(fm*2,  fm,    k=4, s=2, p=1)      → (fm,   32,32)
        #   (fm,   32,32)       → ConvTranspose2d(fm,    channels, k=4, s=2, p=1)   → (ch,   64,64)
        fm = feature_maps
        self.net = nn.Sequential(
            # stage 1: 1×1 → 4×4
            nn.ConvTranspose2d(latent_dim, fm * 8, _______, _______, _______, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.ReLU(inplace=True),
            # stage 2: 4×4 → 8×8
            nn.ConvTranspose2d(fm * 8, _______, _______, _______, _______, bias=False),
            nn.BatchNorm2d(_______),
            nn.ReLU(inplace=True),
            # stage 3: 8×8 → 16×16
            nn.ConvTranspose2d(_______, fm * 2, _______, _______, _______, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.ReLU(inplace=True),
            # stage 4: 16×16 → 32×32
            nn.ConvTranspose2d(fm * 2, _______, _______, _______, _______, bias=False),
            nn.BatchNorm2d(_______),
            nn.ReLU(inplace=True),
            # stage 5: 32×32 → 64×64  (output layer — no BN, Tanh activation)
            nn.ConvTranspose2d(_______, channels, _______, _______, _______, bias=False),
            nn.________(),
        )

    def forward(self, z):
        return self.net(z)


## Build the Discriminator

The discriminator is essentially a CNN classifier: image → convolutional layers → single scalar.  
It uses **LeakyReLU** (not ReLU) to prevent dead neurons, and **no BatchNorm on the input layer**.

Shape flow: `(3,64,64) → (64,32,32) → (128,16,16) → (256,8,8) → (512,4,4) → (1,1,1)`


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=3, feature_maps=64):
        super().__init__()
        fm = feature_maps

        # TODO 29: Complete the discriminator.
        # Use nn.Conv2d(in, out, k=4, s=2, p=1) + nn.BatchNorm2d + nn.LeakyReLU(0.2)
        # No BatchNorm on the FIRST layer (it sees raw pixel values).
        # Final layer: Conv2d → Sigmoid (outputs P(real) in [0, 1])
        #
        # Shape flow:
        #   (ch,    64,64) → (fm,    32,32)   no BN
        #   (fm,    32,32) → (fm*2,  16,16)   BN
        #   (fm*2,  16,16) → (fm*4,   8, 8)   BN
        #   (fm*4,   8, 8) → (fm*8,   4, 4)   BN
        #   (fm*8,   4, 4) → (1,       1, 1)   k=4, s=1, p=0, Sigmoid
        self.net = nn.Sequential(
            # input layer — no BN
            nn.Conv2d(channels, _______, _______, _______, _______, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # stage 2
            nn.Conv2d(_______, fm * 2, _______, _______, _______, bias=False),
            nn.BatchNorm2d(_______),
            nn.LeakyReLU(0.2, inplace=True),
            # stage 3
            nn.Conv2d(_______, fm * 4, _______, _______, _______, bias=False),
            nn.BatchNorm2d(_______),
            nn.LeakyReLU(0.2, inplace=True),
            # stage 4
            nn.Conv2d(_______, fm * 8, _______, _______, _______, bias=False),
            nn.BatchNorm2d(_______),
            nn.LeakyReLU(0.2, inplace=True),
            # output layer — no BN, Sigmoid
            nn.Conv2d(_______, 1, _______, _______, _______, bias=False),
            nn.________(),
        )

    def forward(self, x):
        return self.net(x).view(-1)   # flatten to (batch,)


## Weight Initialisation

DCGAN paper recommends initialising Conv and BN weights from N(0, 0.02).  
This small, consistent initialisation helps training stabilise early on.


In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G = Generator(latent_dim=LATENT_DIM_GAN).to(device)
D = Discriminator().to(device)

G.apply(weights_init)
D.apply(weights_init)

print("Generator parameters : ", sum(p.numel() for p in G.parameters()) / 1e6, "M")
print("Discriminator parameters:", sum(p.numel() for p in D.parameters()) / 1e6, "M")


## Train the GAN

The training loop alternates between two steps each iteration:

1. **Update D**: run real batch + fake batch through D, compute loss, backpropagate into D only.
2. **Update G**: generate a new fake batch, pass through D, compute G loss, backpropagate into G only.

> **Important**: use separate optimisers for G and D. When updating D, don't update G and vice versa.  
> The standard practice is to call `.detach()` on fake images when training D — this prevents gradients  
> from flowing back into G during the discriminator update step.

We track `D_loss`, `G_loss`, and `D(G(z))` (how well G is fooling D) to monitor training health.


In [ ]:
criterion = nn.BCELoss()

# Separate optimisers — typical GAN hyperparameters from the DCGAN paper
d_optimizer = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
g_optimizer = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

REAL_LABEL = 1.0
FAKE_LABEL = 0.0
gan_epochs = 50

d_losses, g_losses = [], []

for epoch in range(1, gan_epochs + 1):
    epoch_d_loss = 0
    epoch_g_loss = 0

    for real_imgs in gan_loader:
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)

        # ── Step 1: Update Discriminator ──────────────────────────────────
        D.zero_grad()

        # Real images — D should output 1
        real_labels = torch.full((batch_size,), REAL_LABEL, device=device)

        # TODO 30: Forward real images through D, compute BCE loss vs real_labels
        d_out_real = D(_______)
        d_loss_real = criterion(_______, _______)

        # Fake images — D should output 0
        noise = torch.randn(batch_size, LATENT_DIM_GAN, 1, 1, device=device)
        fake_imgs = G(noise)
        fake_labels = torch.full((batch_size,), FAKE_LABEL, device=device)

        # TODO 31: Forward fake images through D (detach so G is not updated here).
        # Compute BCE loss vs fake_labels.
        d_out_fake = D(_______.detach())
        d_loss_fake = criterion(_______, _______)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        d_optimizer.step()

        # ── Step 2: Update Generator ───────────────────────────────────────
        G.zero_grad()

        # G wants D to output 1 for its fakes (fool the discriminator)
        # Use the NON-SATURATING loss: -log D(G(z))  i.e. treat fakes as "real" for G's loss
        # TODO 32: Forward the same fake_imgs through D again (no detach this time).
        # Compute BCE loss treating fake_imgs as REAL (use real_labels).
        d_out_for_g = D(_______)
        g_loss = criterion(_______, _______)   # non-saturating: G wants D to say "real"

        g_loss.backward()
        g_optimizer.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()

    d_losses.append(epoch_d_loss / len(gan_loader))
    g_losses.append(epoch_g_loss / len(gan_loader))

    # Show fixed-noise generation every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        G.eval()
        with torch.no_grad():
            samples = G(fixed_noise).cpu()
        G.train()

        # Denormalise from [-1,1] back to [0,1] for display
        samples = (samples * 0.5 + 0.5).clamp(0, 1).numpy().transpose(0, 2, 3, 1)
        fig, axes = plt.subplots(2, 8, figsize=(20, 6))
        for i in range(16):
            axes[i // 8][i % 8].imshow(samples[i])
            axes[i // 8][i % 8].axis('off')
        plt.suptitle(f"Epoch {epoch} — GAN generated dogs")
        plt.tight_layout()
        plt.show()

    print(f"Epoch {epoch}/{gan_epochs}  |  D_loss: {d_losses[-1]:.4f}  |  G_loss: {g_losses[-1]:.4f}")


## Reading the training curves

GAN loss curves behave differently from standard training loss curves — they don't simply go down.

| What you see | What it means |
|---|---|
| D_loss ≈ 0 | D is winning — G produces obvious fakes. G needs more capacity or lower lr. |
| G_loss ≈ 0 | G is winning — D can't distinguish anything. D needs more training. |
| Both losses moderate and stable | Healthy training — both networks pushing each other. |
| G_loss suddenly spikes | Mode collapse — G found one pattern that fools D and collapsed to it. |

In a well-trained GAN, D_loss hovers around `~0.69` (random chance for a binary classifier = log(2)).


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d_losses, label='D loss', color='steelblue')
ax.plot(g_losses, label='G loss', color='tomato')
ax.axhline(0.693, color='gray', linestyle='--', alpha=0.5, label='D random chance (log 2)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('GAN Training Curves')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final D_loss: {d_losses[-1]:.4f}  (healthy target ≈ 0.69)")
print(f"Final G_loss: {g_losses[-1]:.4f}")


## Final generation — sample from trained GAN

Sample fresh random noise vectors and decode with the trained generator.  
These images were never seen during training.


In [ ]:
G.eval()
with torch.no_grad():
    new_noise = torch.randn(32, LATENT_DIM_GAN, 1, 1).to(device)
    new_imgs  = G(new_noise).cpu()

# Denormalise [-1,1] → [0,1]
new_imgs = (new_imgs * 0.5 + 0.5).clamp(0, 1).numpy().transpose(0, 2, 3, 1)

fig = plt.figure(figsize=(20, 8))
for i in range(32):
    ax = fig.add_subplot(4, 8, i + 1, xticks=[], yticks=[])
    plt.imshow(new_imgs[i])
plt.suptitle("GAN generated dogs — fresh random z, no input image used")
plt.tight_layout()
plt.show()


## Side-by-side: VAE vs GAN generation

Generate from both models using the same random seeds so you can directly compare output quality.

> **Expected outcome**: GAN images should appear sharper and more textured.  
> VAE images will be smoother / slightly blurry but more consistent.


In [ ]:
torch.manual_seed(42)

# VAE generation (from Part 2)
vae.eval()
with torch.no_grad():
    vae_z  = torch.randn(8, latent_dim, 4, 4).to(device)
    vae_out = vae.decoder(vae_z).cpu().numpy().transpose(0, 2, 3, 1)

# GAN generation
G.eval()
with torch.no_grad():
    gan_z   = torch.randn(8, LATENT_DIM_GAN, 1, 1).to(device)
    gan_out = ((G(gan_z) * 0.5 + 0.5).clamp(0, 1)).cpu().numpy().transpose(0, 2, 3, 1)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0, i].imshow(vae_out[i].clip(0, 1))
    axes[0, i].axis('off')
    axes[1, i].imshow(gan_out[i])
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("VAE", fontsize=12)
axes[1, 0].set_ylabel("GAN", fontsize=12)
plt.suptitle("VAE vs GAN generation — same dataset, same random seed")
plt.tight_layout()
plt.show()
